In [1]:
!pip -q install fastapi uvicorn python-multipart pymupdf requests gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.0 MB/s eta 0:00:00


In [3]:

!apt-get update -qq
!apt-get install -y zstd


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 77 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 0s (2,591 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [4]:

!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:

import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/",
            timeout=2
        )
        if response.status_code == 200:
            print("Success: Ollama server is running!")
            break
    except requests.RequestException:
        time.sleep(2)
else:
    print("Server may already be running. Check the next cell.")


Success: Ollama server is running!


In [6]:

!ollama pull llama3


In [9]:

import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3",
        "prompt": "Explain PDF summarization in two simple sentences.",
        "stream": False
    },
    timeout=300
)

response.raise_for_status()
print(response.json()["response"])


ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7eb3a9474c30>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [10]:
!ollama --version

In [11]:

import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Server process status:", ollama_process.poll())

try:
    response = requests.get(
        "http://127.0.0.1:11434/",
        timeout=5
    )
    print("HTTP status:", response.status_code)
    print("Response:", response.text)
except requests.RequestException as e:
    print("Connection error:", e)


Server process status: None
HTTP status: 200
Response: Ollama is running


In [12]:

!ollama list


NAME             ID              SIZE      MODIFIED      
llama3:latest    365c0bd3c000    4.7 GB    3 minutes ago    


In [13]:

!ollama pull llama3


In [14]:

import requests

response = requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={
        "model": "llama3",
        "prompt": "Explain PDF summarization in two sentences.",
        "stream": False
    },
    timeout=300
)

response.raise_for_status()
print(response.json()["response"])


PDF summarization is the process of automatically extracting the most important information from a Portable Document Format (PDF) file, such as key phrases, sentences, or concepts, and condensing it into a shorter and more digestible form, often in the form of a summary or abstract. The goal of PDF summarization is to help users quickly grasp the main ideas and takeaways from a lengthy or complex PDF document, such as a research paper, report, or policy brief.


In [16]:

import requests

try:
    response = requests.get(
        "http://127.0.0.1:11434/",
        timeout=5
    )
    print("Status code:", response.status_code)
    print("Response:", response.text)
except requests.RequestException as e:
    print("Connection failed:", e)


Status code: 200
Response: Ollama is running


In [17]:

from google.colab import files

uploaded = files.upload()

pdf_files = [
    name for name in uploaded
    if name.lower().endswith(".pdf")
]

if not pdf_files:
    raise ValueError("Please upload a PDF file.")

pdf_name = pdf_files[0]
pdf_bytes = uploaded[pdf_name]

print("Uploaded successfully:", pdf_name)
print("File size:", round(len(pdf_bytes) / 1024, 2), "KB")


Saving Ashvanth kumar CV.pdf to Ashvanth kumar CV.pdf
Uploaded successfully: Ashvanth kumar CV.pdf
File size: 90.1 KB


In [18]:

!pip -q install pymupdf


In [19]:

import fitz

def extract_pdf_text(pdf_bytes):
    if not pdf_bytes:
        raise ValueError("The PDF file is empty.")

    if len(pdf_bytes) > 15 * 1024 * 1024:
        raise ValueError("PDF size must not exceed 15 MB.")

    try:
        with fitz.open(
            stream=pdf_bytes,
            filetype="pdf"
        ) as pdf:

            if pdf.is_encrypted:
                raise ValueError(
                    "Password-protected PDFs are not supported."
                )

            if len(pdf) == 0:
                raise ValueError("The PDF contains no pages.")

            if len(pdf) > 200:
                raise ValueError("Maximum supported length is 200 pages.")

            pages = []

            for page_number, page in enumerate(pdf, start=1):
                text = page.get_text("text").strip()

                if text:
                    pages.append(
                        f"[Page {page_number}]\n{text}"
                    )

            extracted_text = "\n\n".join(pages)

            if len(extracted_text.strip()) < 50:
                raise ValueError(
                    "Insufficient readable text. "
                    "The PDF may require OCR."
                )

            return extracted_text, len(pdf)

    except (fitz.FileDataError, RuntimeError) as exc:
        raise ValueError("Unable to read this PDF file.") from exc


document_text, page_count = extract_pdf_text(pdf_bytes)

print("PDF extraction successful!")
print("Pages:", page_count)
print("Extracted characters:", len(document_text))
print("\nText preview:\n")
print(document_text[:2000])


PDF extraction successful!
Pages: 1
Extracted characters: 1992

Text preview:

[Page 1]
PROJECTS
SmartIndiaHackathon 2024 | Project
EDUCATION
OBJECTIVE
Enthusiastic and detail-oriented Data Science Student with hands-on experience in Java seeking an
entry-level position in the IT field. Eager to apply my technical skills, problem-solving abilities, and
passion for technology to contribute to the success of a forward-thinking company. To leverage
strong SAP functional/technical skills to deliver efficient enterprise solutions.
 TECHNICAL SKILLS
 
SOFT SKILLS
HOBBY
 
SesquipedaSense – Assistive Ear-Clip Device:
B.Tech | Artificial Intelligence & Data Science
K. Ramakrishnan College of Engineering (Autonomous) 
CGPA - 8.20
Higher Secondary School 
Maxwell Matriculation Higher Secondary School
Percentage - 88%
Secondary School
Maxwell Matriculation Higher Secondary School
Percentage - 92.2%
Thanjavur, Tamilnadu 
ashvanthkumar.j@gmail.com 
https://www.linkedin.com/in/ashvanth-kumar-j-444b36

In [20]:

def split_into_chunks(
    text,
    chunk_size=6000,
    overlap=500
):
    if chunk_size <= overlap:
        raise ValueError(
            "Chunk size must be greater than overlap."
        )

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end == len(text):
            break

        start = end - overlap

    return chunks


chunks = split_into_chunks(document_text)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}: {len(chunk)} characters")
    print(chunk[:200])


Total chunks: 1

Chunk 1: 1992 characters
[Page 1]
PROJECTS
SmartIndiaHackathon 2024 | Project
EDUCATION
OBJECTIVE
Enthusiastic and detail-oriented Data Science Student with hands-on experience in Java seeking an
entry-level position in the I


In [21]:

import requests

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
MODEL_NAME = "llama3"


def ask_llama(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.2
            }
        },
        timeout=300
    )

    response.raise_for_status()

    result = response.json().get("response", "").strip()

    if not result:
        raise RuntimeError("Llama 3 returned an empty response.")

    return result


def summarize_pdf(text):
    chunks = split_into_chunks(text)
    partial_summaries = []

    for i, chunk in enumerate(chunks, start=1):
        prompt = f"""
You are a professional PDF summarization assistant.

Summarize the following document excerpt accurately.
Identify the main purpose, education, projects, skills,
experience, and other important details when present.

Do not invent information.
Do not add information that is absent from the document.
Preserve important names and facts.

DOCUMENT EXCERPT:
{chunk}

Provide a clear summary with headings and bullet points.
"""
        partial_summary = ask_llama(prompt)
        partial_summaries.append(partial_summary)

        print(f"Processed chunk {i} of {len(chunks)}")

    combined_summaries = "\n\n".join(partial_summaries)

    final_prompt = f"""
Create a concise, well-organized summary of this PDF.

Use these headings where relevant:
1. Overview
2. Key Information
3. Important Findings
4. Conclusion

Use only the information in the notes.
Do not invent missing facts or repeat information unnecessarily.

NOTES:
{combined_summaries}
"""

    return ask_llama(final_prompt)


summary = summarize_pdf(document_text)

print("\n" + "=" * 60)
print("FINAL PDF SUMMARY")
print("=" * 60)
print(summary)


Processed chunk 1 of 1

FINAL PDF SUMMARY
**Summary**

**Overview**

This summary provides an overview of Ashvanth Kumar J's education, projects, skills, experience, and achievements.

**Key Information**

* Education: B.Tech in Artificial Intelligence & Data Science from K. Ramakrishnan College of Engineering (Autonomous) with a CGPA of 8.20
* Projects: SesquipedaSense, Online Chatbot Based Ticketing System, Peizoelectric Power Bank, and Wearable device using ENC and ANC technologies
* Skills: Technical skills in Java, HTML, CSS, and SQL, and soft skills in adaptability, time management, and teamwork
* Experience: Participated in Smart India Hackathon 2024 and Smart Innovators Hackathon (2024) - State Level Hackathon
* Achievements: Winner in Paper - Presentation, Selected as Top 10 in Hackathon, and Selected in Top 100 projects of the whole state

**Important Findings**

* Ashvanth Kumar J has a strong educational background in Artificial Intelligence & Data Science
* He has a divers

In [22]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
import fitz
import requests

app = FastAPI(
    title="PDF Summarization Agent",
    description="Summarize PDF documents using Llama 3 and Ollama",
    version="1.0.0"
)


def extract_pdf_text(pdf_bytes):
    if not pdf_bytes:
        raise ValueError("The uploaded PDF is empty.")

    if len(pdf_bytes) > 15 * 1024 * 1024:
        raise ValueError("PDF size must not exceed 15 MB.")

    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as pdf:
            if pdf.is_encrypted:
                raise ValueError("Password-protected PDFs are not supported.")

            if len(pdf) == 0:
                raise ValueError("The PDF contains no pages.")

            if len(pdf) > 200:
                raise ValueError("The PDF must not exceed 200 pages.")

            text = "\n\n".join(
                f"[Page {page_number}]\n{page.get_text()}"
                for page_number, page in enumerate(pdf, start=1)
            )

    except ValueError:
        raise
    except Exception as exc:
        raise ValueError("Invalid or unreadable PDF file.") from exc

    if len(text.strip()) < 50:
        raise ValueError(
            "Not enough readable text found. Scanned PDFs may require OCR."
        )

    return text


@app.get("/")
def home():
    return {
        "message": "PDF Summarization Agent API is running",
        "model": "Llama 3",
        "status": "ready"
    }


@app.post("/summarize")
async def summarize_uploaded_pdf(file: UploadFile = File(...)):
    if not file.filename or not file.filename.lower().endswith(".pdf"):
        raise HTTPException(
            status_code=400,
            detail="Please upload a PDF file."
        )

    try:
        pdf_bytes = await file.read()
        document_text = extract_pdf_text(pdf_bytes)
        summary = summarize_pdf(document_text)

        return {
            "filename": file.filename,
            "characters_extracted": len(document_text),
            "summary": summary
        }

    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))

    except requests.RequestException:
        raise HTTPException(
            status_code=503,
            detail="Unable to connect to Ollama. Check whether Llama 3 is running."
        )

    except Exception as exc:
        raise HTTPException(
            status_code=500,
            detail=f"Summarization failed: {str(exc)}"
        )

    finally:
        await file.close()


print("FastAPI application created successfully!")

FastAPI application created successfully!


In [23]:
import threading
import time
import requests
import uvicorn

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)
api_thread.start()

time.sleep(3)

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

INFO:     Started server process [3217]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:52528 - "GET / HTTP/1.1" 200 OK
Status code: 200
Response: {'message': 'PDF Summarization Agent API is running', 'model': 'Llama 3', 'status': 'ready'}


In [24]:
import requests
from google.colab import files

# Upload a PDF for testing
uploaded_test = files.upload()

pdf_name = next(
    (name for name in uploaded_test if name.lower().endswith(".pdf")),
    None
)

if pdf_name is None:
    print("Please upload a PDF file.")
else:
    pdf_bytes = uploaded_test[pdf_name]

    response = requests.post(
        "http://127.0.0.1:8000/summarize",
        files={
            "file": (
                pdf_name,
                pdf_bytes,
                "application/pdf"
            )
        },
        timeout=600
    )

    print("Status code:", response.status_code)

    if response.ok:
        result = response.json()
        print("\nFilename:", result["filename"])
        print("Characters extracted:", result["characters_extracted"])
        print("\nPDF SUMMARY\n")
        print(result["summary"])
    else:
        print("Error:", response.text)

Saving Ashvanth kumar CV.pdf to Ashvanth kumar CV (1).pdf
Processed chunk 1 of 1
INFO:     127.0.0.1:54692 - "POST /summarize HTTP/1.1" 200 OK
Status code: 200

Filename: Ashvanth kumar CV (1).pdf
Characters extracted: 1993

PDF SUMMARY

Here is a concise, well-organized summary of the PDF:

**Overview**

This summary provides an overview of Ashvanth Kumar J's education, projects, skills, experience, and achievements.

**Key Information**

* Education: B.Tech in Artificial Intelligence & Data Science from K. Ramakrishnan College of Engineering (Autonomous) with a CGPA of 8.20
* Projects: Developed assistive devices, online chatbot systems, and presented projects on Peizoelectric Power Bank and Smart India Hackathon
* Skills: Technical skills in Java, HTML, CSS, SQL, and soft skills in adaptability, time management, and teamwork
* Experience: Participated in hackathons and completed online courses on cloud computing
* Achievements: Won awards and selected as top 10 in hackathons, and pr

In [25]:

import gradio as gr
import requests
import os

API_URL = "http://127.0.0.1:8000/summarize"


def summarize_uploaded_file(pdf_path):
    if not pdf_path:
        return "Please upload a PDF file."

    if not pdf_path.lower().endswith(".pdf"):
        return "Error: Please upload a PDF file."

    try:
        with open(pdf_path, "rb") as pdf_file:
            response = requests.post(
                API_URL,
                files={
                    "file": (
                        os.path.basename(pdf_path),
                        pdf_file,
                        "application/pdf"
                    )
                },
                timeout=600
            )

        if response.status_code == 200:
            result = response.json()

            return (
                f"## PDF Summary\n\n"
                f"**File:** {result['filename']}\n\n"
                f"**Characters extracted:** "
                f"{result['characters_extracted']}\n\n"
                f"---\n\n"
                f"{result['summary']}"
            )

        try:
            error_message = response.json().get(
                "detail", response.text
            )
        except ValueError:
            error_message = response.text

        return f"### Error\n\n{error_message}"

    except requests.exceptions.Timeout:
        return "The summarization took too long. Please try again."

    except requests.exceptions.ConnectionError:
        return (
            "Cannot connect to FastAPI. "
            "Please make sure the API server is running."
        )

    except Exception as error:
        return f"An unexpected error occurred: {error}"


with gr.Blocks(title="PDF Summarization Agent") as demo:
    gr.Markdown(
        """
        # 📄 PDF Summarization Agent

        Upload a PDF document to generate a concise summary
        using **Llama 3** and **FastAPI**.

        **Supported files:** PDF, up to 15 MB and 200 pages.
        """
    )

    pdf_input = gr.File(
        label="Upload your PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    summarize_button = gr.Button(
        "Generate Summary",
        variant="primary"
    )

    summary_output = gr.Markdown(
        label="Generated Summary"
    )

    summarize_button.click(
        fn=summarize_uploaded_file,
        inputs=pdf_input,
        outputs=summary_output
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e043e0517d77f351b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
